# Algorytm symulowanego wyżarzania

Symulowane wyżarzanie (*simulated annealing*) to probabilistyczna technika optymalizacyjna inspirowana procesem wyżarzania w metalurgii. Naśladuje fizyczny proces podgrzewania i powolnego schładzania materiału w celu usunięcia defektów. W kontekście optymalizacji „temperatura” jest skalarnym parametrem algorytmu. 

## Ogólny zarys
Rozpoczyna się od początkowego rozwiązania i wysokiej temperatury, która jest stopniowo obniżana. Na każdym etapie SA losowo modyfikuje bieżące rozwiązanie w celu wygenerowania nowego kandydata. Jeśli nowe rozwiązanie jest lepsze, zostaje zaakceptowane. Jeśli jest gorsze, może również zostać zaakceptowane z pewnym prawdopodobieństwem, które jest proporcjonalne do temperatury. Pozwala to algorytmowi uniknąć lokalnych ekstremów i z czasem przybliżyć rozwiązanie optymalne w skali globalnej.

Tutaj zostanie przedstawiona podstawowa wersja tego algorytmu. Składa się z dwuch pętli. Pierwsza z nich, zewnęczna, zarządza zmianą temperatury. Druga, wewnętrzna, determinuje ile sąsiadów jest sprawdzanych w dla danej temperatury. Poniżej jest zaprezentowany pseudokod: 


1. Inicjalizacja: $s = s_0$, $T = T_{innit}$, wybór funkcji schładzającej $\text{temperature}(n)$
2. For $n \in [0, \ldots, n_{max} - 1]$  
    1. For $k \in [0, \ldots, k_{max} - 1]$   
        1. Wybór sąsiada $s' = \text{neighbour}(s)$
        2. $\beta$ = 1/T
        3. Policzenie $\Delta E = E(s') - E(s)$, $P(\Delta E, \beta)$
        4. Jeżeli $P(\Delta E, \beta)$ > $\text{random}(0,1)$
            * s = s'
    2. $T = \text{temperature}(n)$
3. Zwróć $s$, $E(s)$


Jako funkcji $P(\Delta E, \beta)$ użyjemy tzw. kryterium Metropolisa–Hastingsa:
$$
P(\Delta E, \beta) = \left\{
                            \begin{array}{ll}
                            1, & \text{jeżeli } \Delta E \leq 0 \\
                            e^{-\beta \Delta E}, & \text{w pozostałych przypadkach}
                            \end{array}
                            \right.
$$


Warto tutaj wspomnieć, że dla "wystarczająco powolnej" funkcji schładzającej istnieją matematyczne gwarancje osiągnięcia optymalnego wyniku. Jednakże czas i ilość kroków potrzebna by tą gwarancję osiągnąć często przekracza wyczerpujące przeszukiwanie.


In [13]:
# Implementacja algorytmu symulowanego wyżarzania

import numpy as np
from math import exp
from funkcje_pomocnicze import calculate_energy

def random_neightbour(conf: np.ndarray):
    flip_idx = np.random.randint(len(conf))
    conf[flip_idx] *= -1  # zmieniamy spin losowego wierzchołka
    return conf


def fixed_spin(conf: np.ndarray, idx: int):
    conf[idx] *= -1
    return(conf)


def acceptance_probability(delta_e: float, temp: float):
    if delta_e < 0:
        return 1
    else:
        beta = 1/temp
        probability = exp(-beta * delta_e)
        
        return probability

def simulated_annealing_fixed_steps(J, h, initial_temp: float, num_steps: int, num_repeats: int = 1, schedule: str = "linear",):
    n = len(h)
    solution = np.random.choice([-1, 1], size=n)
    energy = calculate_energy(J, h, solution)

    T_0 = initial_temp
    T_final = 1e-12

    if schedule == "linear":
        schedule = np.linspace(T_0, T_final, num=num_steps, endpoint=True)

    elif schedule == "geometric":
        alpha = 0.95
        schedule = np.array([max(T_0 * (alpha ** i), T_final) for i in range(num_steps)])

    elif schedule == "exponential":
        schedule = np.geomspace(T_0, T_final, num=num_steps, endpoint=True)

    for k in range(num_steps):
        temp = schedule[k]
        for _ in range(num_repeats):
    
            new_solution = random_neightbour(solution)
            new_energy = calculate_energy(J, h, new_solution)
            delta_e = new_energy - energy

            r = np.random.random()
            
            if acceptance_probability(delta_e, temp) > r:
                solution = new_solution
                energy = new_energy


    return solution, energy
 


In [38]:
# Test dla małej instancji

from funkcje_pomocnicze import read_instance, small_grid, test_pegasus
from tqdm import tqdm

instance = small_grid

J, h = read_instance(instance.path)

# Ponieważ jest to probabilistyczny algorytm warto go puścić kilka razy i wybrać najlepszy wynik
energies = []
for _ in tqdm(range(10), desc="wielokrone symulowane wyżarzanie"):
    state, energy = simulated_annealing_fixed_steps(J, h, initial_temp=10, num_steps=100, num_repeats=100, schedule="geometric")
    energies.append(energy)

print(f"Otrzymana energia: {min(energies)}")
print(f"Najniższa znana energia: {instance.best_energy}")
print(f"Luka energetyczna: {((instance.best_energy - min(energies))/instance.best_energy * 100):.2f}%")

wielokrone symulowane wyżarzanie: 100%|██████████| 10/10 [00:01<00:00,  8.31it/s]

Otrzymana energia: -20.75
Najniższa znana energia: -22.75
Luka energetyczna: 8.79%


## Rozbudowa

Pokazana tu implementacja jest bardzo podstawowa, przez co nie osiąga zadawalających rezultatów. Istnieje w literaturze wiele, często wyrafinowanych rozszerzeń tego algorytmu. Współcześnie wykorzystuje się adaptacyje schematy (funkcje) chłodzenia, ponowne nagrzewanie i inne techniki. Dobrze zoptymalizowane symulowane wyrzażanie z odpowiednio dobranymi parametrami ciągle pozostaje konkurencyją metodą rozwiązywania problemów QUBO.

In [ ]:
# Przykład użycia zoptymalizowanej implementacji

from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

from funkcje_pomocnicze import small_grid, read_instance, test_pegasus, Grid100

instance = Grid100 # Siatka kwadratowa rozmiaru 100 x 100

J2, h2 = read_instance(instance.path, convention="dwave")

bqm_instance = BinaryQuadraticModel(h2, J2, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=10)
energy = sampleset.first.energy


print(f"Otrzymana energia: {energy}")
print(f"Najniższa znana energia: {instance.best_energy}")
print(f"Luka energetyczna: {((instance.best_energy - energy)/instance.best_energy * 100):.2f}%")


Otrzymana energia: -10537.25
Najniższa znana energia: -10548.25
Luka energetyczna: 0.10%


# Bibliografia

* Eglese, R.W., Simulated annealing: A tool for operational research. *European Journal of Operational Research*, Volume **46**, Issue **3**, 1990, DOI: [10.1016/0377-2217(90)90001-R](https://doi.org/10.1016/0377-2217(90)90001-R)

* Henderson, D., Jacobson, S.H., Johnson, A.W. (2003). The Theory and Practice of Simulated Annealing. In: Glover, F., Kochenberger, G.A. (eds) Handbook of Metaheuristics. International Series in Operations Research & Management Science, vol 57. Springer, Boston, MA. DOI: [10.1007/0-306-48056-5_10](https://doi.org/10.1007/0-306-48056-5_10)

* Vodeb, J., Eržen, V., Hrga, T., Povh, J. (2024). Accuracy and Performance Evaluation of Quantum, Classical and Hybrid Solvers for the Max-Cut Problem. *ArXiv preprint*, DOI: [10.48550/arXiv.2412.07460](https://doi.org/10.48550/arXiv.2412.07460)
